***[May 22]***

Idea:

- Turn each meal into Darts' series with time as covariate
- Each restaurant uses a global model to forecast all meals within that restaurant

Results:
- Solely compare with old method (aka `idea9_May14`) on Chemicum, the old method still performs better

In [1]:
%cd ../../
%env PYTHONWARNINGS=ignore

/Users/hoangle/Projects/untangling-people/ylva/fwo_models
env: PYTHONWARNINGS=ignore


In [2]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import pandas as pd
from darts.timeseries import concatenate
import polars as pl
from darts.models import NaiveMovingAverage, LinearRegressionModel, RandomForest, CatBoostModel, KalmanForecaster
from darts import TimeSeries
from darts.dataprocessing.transformers import Scaler, StaticCovariatesTransformer, MissingValuesFiller, InvertibleMapper
from darts.dataprocessing import Pipeline
from darts.metrics import mape
from tqdm import tqdm

In [3]:
plt.style.use('seaborn-v0_8')
plt.rcParams.update({'font.size': 8})

# Suppress warnings globally
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [4]:
CUTOFF_DATE = "2025-01-01"
LAG = 2
MIN_TRAIN_SIZE = LAG + 1

# Load dims and raw data

## dim `meals`

In [5]:
dim_meals = pl.read_parquet("data/processed/dim_meals.parquet")
dim_meals.head()

id,meal_codes,names,restaurants,meal_type,schoolyear,attributes,co2,src
i64,list[i64],list[str],list[i64],i64,str,list[str],f32,list[str]
0,[90000000],"[""Kikhernetaginea& syysomenajogurttia""]",[1],5,"""23-24""",[],0.41,"[""pos_Jan23-Oct24""]"
1,[90000001],"[""Rapea Meiramikana""]","[1, 4]",3,"""23-24""",[],1.26,"[""pos_Jan23-Oct24""]"
2,[7203],"[""Kasvismuhennos Caponata""]","[1, 2, 4]",4,"""23-24""",[],0.42,"[""pos_Jan23-Oct24"", ""menus_meals""]"
3,[9058],"[""TexMex-siemenpyöryköitä ja Arrabiattakastiketta"", ""TexMex-siemenpyöryköitä ja Arrabiattakastiket""]","[1, 2, 4]",4,"""24-25""","[""gluten_free"", ""vegan-kpl""]",0.56,"[""pos_Jan23-Oct24"", ""menus_meals"", … ""menus_week1-6""]"
4,[6877],"[""Kasvisjalapenonuggetteja, tomaattisalsaa"", ""Kasvis-jalapnuget ja tomatsals""]","[1, 2, 4]",4,"""24-25""",[],0.44,"[""pos_Jan23-Oct24"", ""menus_meals"", ""pos_Nov24-Mar25""]"


## dim `meal_types`

In [6]:
path = "data/processed/dim_meal_types.xlsx"
dim_meal_types = pl.read_excel(path)

dim_meal_types.head()

meal_type_id,meal_type,meal_type_en
i64,str,str
1,"""Kala""","""fish"""
2,"""Liha""","""meat"""
3,"""Kana""","""chicken"""
4,"""Vegaani""","""vegan"""
5,"""Kasvis""","""vegetarian"""


## dim `exams_uhelsinki`

In [7]:
path = "data/processed/dim_exams_uhelsinki.xlsx"
dim_exams_uhelsinki = pl.read_excel(path)

dim_exams_uhelsinki.head()

date,is_exam
date,bool
2023-01-01,false
2023-01-02,false
2023-01-03,false
2023-01-04,false
2023-01-05,false


## dim `holidays_uhelsinki`

In [8]:
path = "data/processed/dim_holidays_uhelsinki.xlsx"

dim_holidays_uhelsinki = pl.read_excel(path)
dim_holidays_uhelsinki.head()

date,is_holiday
date,bool
2023-01-01,true
2023-01-02,false
2023-01-03,false
2023-01-04,false
2023-01-05,false


## fact `pos`

In [9]:
path = "data/processed/pos.xlsx"
pos_raw = pl.read_excel(path)

pos_raw.head()

id,restaurant,meal_id,datetime,pcs,src
i64,i64,i64,datetime[ms],i64,str
0,1,55,2023-01-02 10:31:00,1,"""Sold lunches"""
1,1,8,2023-01-02 10:32:00,1,"""Sold lunches"""
2,1,55,2023-01-02 10:32:00,1,"""Sold lunches"""
3,1,8,2023-01-02 10:35:00,1,"""Sold lunches"""
4,1,55,2023-01-02 10:36:00,2,"""Sold lunches"""


In [10]:
pos = (
    pos_raw
    .with_columns(
        pl.col('datetime').dt.date().alias('date')
    )
    .group_by('restaurant', 'date', 'meal_id')
    .agg(pl.col('pcs').sum())


    # Add info for holiday and exam
    .join(dim_holidays_uhelsinki, on='date', how='left')
    .join(dim_exams_uhelsinki, on='date', how='left')
    .with_columns(
        pl.col('is_holiday').cast(pl.Int32),
        pl.col('is_exam').cast(pl.Int32)
    )


    # Add meal_type
    .join(
        dim_meals.select(pl.col('id').alias('meal_id'), 'meal_type'),
        on='meal_id', how='left'
    )
)


# Only keep entries of meals having data since CUTOFF_DATE
# meals_concerned = (
#     pos
#     .filter(pl.col('date') >= pl.lit(CUTOFF_DATE, dtype=pl.Date))
#     .select('meal_id', 'restaurant').unique()
# )

# pos = pos.join(meals_concerned, on=['meal_id', 'restaurant'], how='inner')


# Remove pos of leftover meals
pos = (
    pos
    .with_columns(
        pl.col('date').shift(1).over('restaurant', 'meal_id', order_by='date').alias('date_lag'),
        pl.col('pcs').shift(1).over('restaurant', 'meal_id', order_by='date').alias('pcs_lag'),
    )
    .filter(
        (pl.col('date_lag').is_null())
        | ((pl.col('date') - pl.col('date_lag')).dt.total_days() > 7)
    )
    .drop('date_lag', 'pcs_lag')
)


# Keep records whose POS value is greater than Q1 value
pos = (
    pos
    .with_columns(
        pl.col('pcs').quantile(.25).over('restaurant', 'meal_id', order_by='date').alias('pcs_q1'),
        pl.col('pcs').quantile(.75).over('restaurant', 'meal_id', order_by='date').alias('pcs_q3')
    )
    .filter(
        (1 == 1)
        & (pl.col('pcs') >= pl.col('pcs_q1'))
        & (pl.col('pcs') <= pl.col('pcs_q3'))
    )
    .drop('pcs_q1', 'pcs_q3')
)



# Add datetime attributes
pos = (
    pos
    .with_columns(
        pl.col('date').dt.weekday().alias('weekday'),
        pl.col('date').dt.day().alias('day'),
        pl.col('date').dt.week().alias('week'),
        pl.col('date').dt.month().alias('month'),
        pl.col('date').dt.year().alias('year'),
    )
    .with_columns(
        (pl.col('weekday') * 2 * np.pi / 7).sin().alias('weekday_sin'),
        (pl.col('weekday') * 2 * np.pi / 7).cos().alias('weekday_cos'),
        (pl.col('day') * 2 * np.pi / 31).sin().alias('day_sin'),
        (pl.col('day') * 2 * np.pi / 31).cos().alias('day_cos'),
        (pl.col('week') * 2 * np.pi / 53).sin().alias('week_sin'),
        (pl.col('week') * 2 * np.pi / 53).cos().alias('week_cos'),
        (pl.col('month') * 2 * np.pi / 12).sin().alias('month_sin'),
        (pl.col('month') * 2 * np.pi / 12).cos().alias('month_cos'),
    )
)


pos.head()

restaurant,date,meal_id,pcs,is_holiday,is_exam,meal_type,weekday,day,week,month,year,weekday_sin,weekday_cos,day_sin,day_cos,week_sin,week_cos,month_sin,month_cos
i64,date,i64,i64,i32,i32,i64,i8,i8,i8,i8,i32,f64,f64,f64,f64,f64,f64,f64,f64
1,2024-04-17,192,226,0,0,1,3,17,16,4,2024,0.433884,-0.900969,-0.299363,-0.954139,0.947326,-0.32027,0.866025,-0.5
4,2024-03-13,78,69,0,0,2,3,13,11,3,2024,0.433884,-0.900969,0.485302,-0.874347,0.964636,0.263587,1.0,6.1232e-17
2,2024-09-02,214,1,0,0,2,1,2,36,9,2024,0.781831,0.62349,0.394356,0.918958,-0.902798,-0.430065,-1.0,-1.8370e-16
4,2025-01-14,247,234,0,0,4,2,14,3,1,2025,0.974928,-0.222521,0.299363,-0.954139,0.348202,0.93742,0.5,0.866025
1,2024-12-18,533,60,0,1,4,3,18,51,12,2024,0.433884,-0.900969,-0.485302,-0.874347,-0.234886,0.972023,-2.4493e-16,1.0


# Build Time series

In [55]:
restaurant = 1
meal_type = 5

In [56]:
col_tgt = 'pcs'

cols_cov = [
    'is_exam',
    'is_holiday',

    'weekday',
    'day',
    'week',
    'month',
    # 'year',

    'weekday_sin',
    'weekday_cos',
    'day_sin',
    'day_cos',
    'week_sin',
    'week_cos',
    'month_sin',
    'month_cos',
]

### Build dict `meals` as follow

```
meals = {
    '<meal_id>': {
        'train_tgt': TimeSeries| None,
        'train_cov': TimeSeries| None,
        'test_tgt': TimeSeries | None,
        'test_cov': TimeSeries| None,
        'trainable': bool,
    }
}
```

In [57]:
meals = {}

meal_ids = (
    pos
    .filter(
        (1 == 1)
        & (pl.col('restaurant') == restaurant)
        & (pl.col('meal_type') == meal_type)
    )
    ['meal_id']
    .unique()
)
for meal_id in meal_ids:
    df = (
        pos
        .filter(
            (1 == 1)
            & (pl.col('meal_id') == meal_id)
            & (pl.col('restaurant') == restaurant)
            & (pl.col('meal_type') == meal_type)
        )
        .sort('date')
    )

    cutoff = len(df.filter(pl.col('date') < pl.lit(CUTOFF_DATE).str.to_date()))
    tgt = TimeSeries.from_dataframe(df, value_cols=col_tgt, fillna_value=False)
    cov = TimeSeries.from_dataframe(df, value_cols=cols_cov, fillna_value=False)

    test_tgt, test_cov = None, None
    if cutoff < len(tgt):
        train_tgt, test_tgt = tgt.split_before(cutoff)
        train_cov, test_cov = cov.split_before(cutoff)
    else:
        train_tgt, train_cov = tgt, cov

    meals[meal_id] = {
        'train_tgt': train_tgt,
        'train_cov': train_cov,
        'test_tgt': test_tgt,
        'test_cov': test_cov,
        'trainable': len(train_tgt) >= MIN_TRAIN_SIZE,
        'testable': len(train_tgt) >= MIN_TRAIN_SIZE and test_tgt is not None and len(test_tgt) > 0
    }


# df.head()

No time column specified (`time_col=None`) and no index found in the `DataFrame`. Defaulting to `pandas.RangeIndex(len(df))`. If this is not desired consider adding a time column to your `DataFrame` and defining `time_col`.
No time column specified (`time_col=None`) and no index found in the `DataFrame`. Defaulting to `pandas.RangeIndex(len(df))`. If this is not desired consider adding a time column to your `DataFrame` and defining `time_col`.
No time column specified (`time_col=None`) and no index found in the `DataFrame`. Defaulting to `pandas.RangeIndex(len(df))`. If this is not desired consider adding a time column to your `DataFrame` and defining `time_col`.
No time column specified (`time_col=None`) and no index found in the `DataFrame`. Defaulting to `pandas.RangeIndex(len(df))`. If this is not desired consider adding a time column to your `DataFrame` and defining `time_col`.
No time column specified (`time_col=None`) and no index found in the `DataFrame`. Defaulting to `pandas.

# Train

## Build Scaler

In [58]:
df_train = pos.filter(
    (1 == 1)
    & (pl.col('restaurant') == restaurant)
    & (pl.col('meal_type') == meal_type)
    & (pl.col('date') < pl.lit(CUTOFF_DATE).str.to_date()) 
)

tgt = TimeSeries.from_dataframe(df, value_cols=col_tgt, fillna_value=False)
cov = TimeSeries.from_dataframe(df, value_cols=cols_cov, fillna_value=False)


No time column specified (`time_col=None`) and no index found in the `DataFrame`. Defaulting to `pandas.RangeIndex(len(df))`. If this is not desired consider adding a time column to your `DataFrame` and defining `time_col`.
No time column specified (`time_col=None`) and no index found in the `DataFrame`. Defaulting to `pandas.RangeIndex(len(df))`. If this is not desired consider adding a time column to your `DataFrame` and defining `time_col`.


In [59]:
pipeline_tgt = Pipeline([
    # InvertibleMapper(np.log1p, np.expm1, verbose=False, n_jobs=-1, name="Log-Transform"),
    Scaler(verbose=False, n_jobs=-1, name="Scaling"),
])
pipeline_tgt.fit(tgt)


pipeline_cov = Pipeline([
    Scaler(verbose=False, n_jobs=-1, name="Scaling"),
])
pipeline_cov.fit(cov)

## Collect series for training and testing (including target and covariate)

In [60]:
train_meal_ids, train_tgt, train_cov  = [], [], []
test_meal_ids = []

for meal_id, meal in meals.items():
    if meal['trainable']:
        train_meal_ids.append(meal_id)
        train_tgt.append(pipeline_tgt.transform(meal['train_tgt']))
        train_cov.append(pipeline_cov.transform(meal['train_cov']))

    if meal['testable']:
        test_meal_ids.append(meal_id)

## Train

In [61]:
model_name = "rf"

# model = LinearRegressionModel(LAG, lags_future_covariates=[0])
# model.fit(train_tgt, future_covariates=train_cov)


match model_name:
    case "linear":
        model = LinearRegressionModel(LAG, lags_future_covariates=[0])
        model.fit(train_tgt, future_covariates=train_cov)
    case "rf":
        model = RandomForest(LAG, lags_future_covariates=[0])
        model.fit(train_tgt, future_covariates=train_cov)

## Predict

In [62]:
predictions = []
for meal_id in test_meal_ids:
    meal = meals[meal_id]

    train_tgt  = pipeline_tgt.transform(meal['train_tgt'])
    test_cov  = pipeline_cov.transform(meal['test_cov'])
    test_tgt = meal['test_tgt']

    match model:
        case LinearRegressionModel():
            out = model.predict(len(test_tgt), series=train_tgt, future_covariates=test_cov)
        case RandomForest():
            out = model.predict(len(test_tgt), series=train_tgt, future_covariates=test_cov)

    test_tgt_pred = pipeline_tgt.inverse_transform(out)
    assert isinstance(test_tgt_pred, TimeSeries)

    for pred, tgt in zip(
        test_tgt_pred.to_dataframe()[col_tgt].to_list(),
        test_tgt.to_dataframe()[col_tgt].to_list()
    ):
        predictions.append({
            'meal_id': meal_id,
            'restaurant':  restaurant,
            'meal_type': meal_type,
            'pred': pred,
            'tgt': tgt
        })

# model.predict()

In [63]:
path = f"data/inter/evaluation/per_meal/{restaurant}_{meal_type}.xlsx"

pd.DataFrame.from_records(predictions).to_excel(path, index=False)